## Generating summary & bias for each article:

In [7]:
sys.path.append("/data/cb/scratch/bfefferm/NLP-Project/hfppl")

In [25]:
import sys
import pandas as pd
import csv
import os
from hfppl import Model, LMContext, TokenCategorical, CachedCausalLM, smc_steer, smc_standard
from score import compute_pbf_score, compute_pbi_score
from smc_steer import bias_model_factory,TwistModel, gen_summary

**Loading Dataset:**

In [26]:
dataset = pd.read_csv('../POLITICS_finetuning/processed_data.csv')

In [27]:
dataset

,title,body,stance
0,"Ryan goes on offense over Medicare, accuses Ob...",Paul Ryan went on offense Tuesday in response ...,center
1,Obama Medicare Attack In 2008 Targeted McCain ...,WASHINGTON -- In the wake of Mitt Romney's cho...,right
2,Clintons Report Earnings of $139 Million in Se...,Hillary Rodham Clinton on Friday released her ...,left
3,Clinton camp releases candidate's clean bill o...,WASHINGTON – Hillary Clinton's presidential ca...,center
4,Elizabeth Cheney blasted by older sister over ...,"Call it Cheney versus Cheney.\nMary Cheney, on...",center
...,...,...,...
295,Chris Christie Announces He's Running For Pres...,Chris Christie Announces He's Running For Pres...,right
296,"Bloomberg says Trump a 'dangerous demagogue,' ...",Former New York City Mayor Michael Bloomberg f...,conservative
297,Donald Trump Says Joe Lieberman Is His Top Cho...,WASHINGTON ― President Donald Trump is “very c...,liberal
298,Trump's 'Compromise' Immigration Offer To Demo...,WASHINGTON ― President Donald Trump’s offer to...,liberal


**Specifying file paths:**

In [28]:
# Model paths:
bias_model_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/politics_best_2500_30bz_000007_best'
bias_tokenizer_path = '/data/cb/scratch/bfefferm/NLP-Project-storage/Saved_Models/politics_best_2500/tokenizer_politics_best_2500_30bz_000007_best'

bias_model = bias_model_factory(bias_model_path, bias_tokenizer_path)

# Specifying model name:
llm_model_name = 'facebook/bart-large-cnn'

# Loading llm:
llm = CachedCausalLM.from_pretrained(llm_model_name)

Some weights of the model checkpoint at facebook/bart-large-cnn were not used when initializing BartForCausalLM: ['model.encoder.layers.4.fc1.weight', 'model.encoder.layers.4.self_attn_layer_norm.weight', 'model.encoder.layers.6.fc2.weight', 'model.encoder.layers.11.self_attn.k_proj.weight', 'model.encoder.layers.4.self_attn.out_proj.weight', 'model.encoder.layers.4.fc2.weight', 'model.encoder.layers.6.self_attn_layer_norm.weight', 'model.encoder.layers.7.self_attn.out_proj.weight', 'model.encoder.layers.3.fc2.weight', 'model.encoder.layers.1.final_layer_norm.bias', 'model.encoder.layers.10.self_attn.out_proj.weight', 'model.encoder.layers.1.self_attn.k_proj.bias', 'model.encoder.layers.6.final_layer_norm.weight', 'model.encoder.layers.3.self_attn.v_proj.weight', 'model.encoder.layers.8.self_attn_layer_norm.bias', 'model.encoder.layers.11.fc2.weight', 'model.encoder.layers.6.self_attn_layer_norm.bias', 'model.encoder.layers.2.self_attn.k_proj.weight', 'model.encoder.layers.2.self_attn_

**Iterating over each summary, predicting bias, & saving to `.csv`:**

In [30]:
summaryAndArticle_list = []
res_list =  [] # List to which one can write results

# Open file  
with open('../POLITICS_finetuning/processed_data.csv') as file_obj: 
      
    # Create reader object by passing the file  
    # object to reader method 
    reader_obj = csv.reader(file_obj) 

    # The fields of this file are
    # ['title', 'body', 'stance']
    
    # Iterate over each row in the .csv,
    # skipping the first row (pertaining to field / column)
    next(reader_obj)
    for row in reader_obj: 
        # Store title:
        title = row[0]
        # Store article:
        article = row[1]
        # Ensuring that articles fit within maximum length
        article = article[:llm.tokenizer.model_max_length] 
        # Store stance:
        stance = row[2]
        # Generate summary:
        summary = await gen_summary(llm_model_name, llm, bias_model, TwistModel, article, stance)
        # For each summary, predict its bias:
        pred_bias, logits = bias_model(article)
        # Append to list of biases:
        summaryAndArticle_list.append([title, summary, pred_bias])

reader_obj.close()
    
with open('../POLITICS_finetuning/summariesAndBiases.csv', 'w') as f: 
    # using csv.writer method from CSV package
    writer_obj = csv.writer(f)
    writer_obj.writerows(summaryAndArticle_list)

writer_obj.close()

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
